In [ ]:
import pandas as pd
import time
# Load the CSV file
df = pd.read_csv("smart_logistic_tracker_japan.csv")

# Display the first 5 rows
df.head()

In [ ]:
from web3 import Web3

# Connect to local blockchain
ganache_url = "http://127.0.0.1:8545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

# Verify connection
if web3.is_connected():
    print("✅ Connected to Ganache successfully!")
else:
    print("❌ Connection failed. Ensure Ganache is running.")

In [ ]:
# Replace with actual contract address from Remix
contract_address = "0x61E6dcad16A78579490402c32f6947E4a68bf838"


# Paste the ABI from Remix
abi = [
  {
    "inputs": [],
    "stateMutability": "nonpayable",
    "type": "constructor"
  },
  {
    "anonymous": False,
    "inputs": [
      {
        "indexed": False,
        "internalType": "uint256",
        "name": "timestamp",
        "type": "uint256"
      },
      {
        "indexed": False,
        "internalType": "string",
        "name": "packageId",
        "type": "string"
      },
      {
        "indexed": False,
        "internalType": "string",
        "name": "currentLocation",
        "type": "string"
      },
      {
        "indexed": False,
        "internalType": "string",
        "name": "status",
        "type": "string"
      }
    ],
    "name": "StatusUpdated",
    "type": "event"
  },
  {
    "inputs": [
      {
        "internalType": "string",
        "name": "_packageId",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "_location",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "_status",
        "type": "string"
      }
    ],
    "name": "storeStatus",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "index",
        "type": "uint256"
      }
    ],
    "name": "getPackageUpdate",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "",
        "type": "string"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "getTotalRecords",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "name": "logisticsRecords",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "timestamp",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "packageId",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "currentLocation",
        "type": "string"
      },
      {
        "internalType": "string",
        "name": "status",
        "type": "string"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "MAX_ENTRIES",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "owner",
    "outputs": [
      {
        "internalType": "address",
        "name": "",
        "type": "address"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  }
]  # Replace with your contract ABI

# Load the smart contract
contract = web3.eth.contract(address=contract_address, abi=abi)

# Set the default sender address (first account from Ganache)
web3.eth.default_account = web3.eth.accounts[0]

print(f"✅ Connected to Smart Contract at {contract_address}")

In [ ]:
def send_iot_data(package_id, location, status):
    """
    Sends logistics IoT data
    to the deployed smart contract
    """

    txn = contract.functions.storeStatus(
        package_id,
        location,
        status
    ).transact({
        'from': web3.eth.default_account,
        'gas': 3000000
    })

    # Wait for transaction confirmation
    receipt = web3.eth.wait_for_transaction_receipt(txn)

    print(
        f"✅ Data Stored | {package_id} | "
        f"Location: {location} | "
        f"Status: {status} | "
        f"Txn Hash: {receipt.transactionHash.hex()}"
    )

# Store 100 records
for index, row in df.head(100).iterrows():

    package_id = str(row["package_id"])
    location = str(row["current_location"])
    status = str(row["latest_status"])

    send_iot_data(package_id, location, status)

    # Delay between transactions
    time.sleep(1)

print("\n✅ Successfully stored 100 records on the blockchain!")

In [ ]:
current_records = contract.functions.getTotalRecords().call()
print(f"Total IoT records stored: {current_records}")

In [ ]:
# Retrieve and display the first stored record
first_record = contract.functions.getPackageUpdate(0).call()

print("📦 First Stored Record")
print(f"Timestamp: {first_record[0]}")
print(f"Package ID: {first_record[1]}")
print(f"Location: {first_record[2]}")
print(f"Status: {first_record[3]}")